In [1]:
import gymnasium as gym
import random, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

# --- Q Network ---
class QNet(nn.Module):
    def __init__(self, s, a):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(s,128), nn.ReLU(), nn.Linear(128,a))
    def forward(self,x): return self.fc(x)

# --- Replay Buffer ---
class RB:
    def __init__(self, size=10000): self.buf=deque(maxlen=size)
    def push(self,*x): self.buf.append(x)
    def sample(self,n):
        b = random.sample(self.buf,n)
        return map(lambda t: torch.tensor(np.array(t),dtype=torch.float32), zip(*b))
    def __len__(self): return len(self.buf)

# --- Setup ---
env = gym.make("CartPole-v1")
s_dim, a_dim = env.observation_space.shape[0], env.action_space.n
dev = "cuda" if torch.cuda.is_available() else "cpu"

policy = QNet(s_dim,a_dim).to(dev)
target = QNet(s_dim,a_dim).to(dev)
target.load_state_dict(policy.state_dict())

opt = optim.Adam(policy.parameters(), lr=1e-3)
buf = RB()
γ = 0.99
ε, ε_min, ε_decay = 1.0, 0.01, 0.995
B = 64

# --- Training ---
for ep in range(400):
    s,_ = env.reset()
    total = 0
    for _ in range(500):
        # ε-greedy
        a = env.action_space.sample() if random.random()<ε \
            else policy(torch.tensor(s).float().to(dev)).argmax().item()

        s2,r,done,trunc,_ = env.step(a)
        buf.push(s,a,r,s2,float(done))
        s = s2; total += r

        # Train
        if len(buf)>B:
            S,A,R,S2,D = buf.sample(B)
            S,S2 = S.to(dev), S2.to(dev)
            A = A.long().unsqueeze(1).to(dev)
            R,D = R.unsqueeze(1).to(dev), D.unsqueeze(1).to(dev)

            q = policy(S).gather(1,A)
            with torch.no_grad():
                y = R + γ * target(S2).max(1)[0].unsqueeze(1) * (1-D)

            opt.zero_grad()
            nn.SmoothL1Loss()(q,y).backward()
            opt.step()

        if done or trunc: break

    ε = max(ε_min, ε*ε_decay)
    if ep % 10 == 0: target.load_state_dict(policy.state_dict())
    print(f"Ep {ep+1}, R={total}, ε={ε:.3f}")

# --- Test ---
scores=[]
for _ in range(10):
    s,_=env.reset(); tot=0
    while True:
        a = policy(torch.tensor(s).float().to(dev)).argmax().item()
        s,r,done,trunc,_ = env.step(a)
        tot+=r
        if done or trunc: break
    scores.append(tot)
    print("Test:", tot)

print("Avg:", np.mean(scores))
env.close()

Ep 1, R=28.0, ε=0.995
Ep 2, R=16.0, ε=0.990
Ep 3, R=13.0, ε=0.985
Ep 4, R=34.0, ε=0.980
Ep 5, R=20.0, ε=0.975
Ep 6, R=34.0, ε=0.970
Ep 7, R=33.0, ε=0.966
Ep 8, R=10.0, ε=0.961
Ep 9, R=8.0, ε=0.956
Ep 10, R=15.0, ε=0.951
Ep 11, R=12.0, ε=0.946
Ep 12, R=14.0, ε=0.942
Ep 13, R=10.0, ε=0.937
Ep 14, R=14.0, ε=0.932
Ep 15, R=18.0, ε=0.928
Ep 16, R=14.0, ε=0.923
Ep 17, R=30.0, ε=0.918
Ep 18, R=14.0, ε=0.914
Ep 19, R=42.0, ε=0.909
Ep 20, R=31.0, ε=0.905
Ep 21, R=30.0, ε=0.900
Ep 22, R=18.0, ε=0.896
Ep 23, R=19.0, ε=0.891
Ep 24, R=18.0, ε=0.887
Ep 25, R=22.0, ε=0.882
Ep 26, R=14.0, ε=0.878
Ep 27, R=17.0, ε=0.873
Ep 28, R=30.0, ε=0.869
Ep 29, R=22.0, ε=0.865
Ep 30, R=11.0, ε=0.860
Ep 31, R=25.0, ε=0.856
Ep 32, R=14.0, ε=0.852
Ep 33, R=17.0, ε=0.848
Ep 34, R=41.0, ε=0.843
Ep 35, R=15.0, ε=0.839
Ep 36, R=28.0, ε=0.835
Ep 37, R=23.0, ε=0.831
Ep 38, R=13.0, ε=0.827
Ep 39, R=13.0, ε=0.822
Ep 40, R=20.0, ε=0.818
Ep 41, R=30.0, ε=0.814
Ep 42, R=42.0, ε=0.810
Ep 43, R=24.0, ε=0.806
Ep 44, R=22.0, ε=0.80